In [9]:
import pandas as pd

In [10]:
df1 = pd.read_csv('C:/Users/human/미정프로젝트/서울시 일반음식점 인허가 정보.csv', encoding = 'cp949',
                  encoding_errors='replace', low_memory = False)
df2 = pd.read_csv('C:/Users/human/미정프로젝트/서울시 휴게음식점 인허가 정보.csv', encoding = 'cp949',
                  encoding_errors='replace', low_memory = False)

print(f'일반음식점: {len(df1):,}행')
print(f'휴게음식점: {len(df2):,}행')

일반음식점: 532,915행
휴게음식점: 144,924행


In [11]:
df1['업종구분'] = '일반음식점'
df2['업종구분'] = '휴게음식점'

df = pd.concat([df1, df2], ignore_index=True)
print(f'합친 행 수:{len(df):,}')
print(df['영업상태명'].value_counts())

합친 행 수:677,839
영업상태명
폐업       520384
영업/정상    157455
Name: count, dtype: int64


In [12]:
# 필요한 컬럼만 추출
cols = ['사업장명', '업태구분명', '업종구분', '영업상태명', 
        '인허가일자', '폐업일자', '도로명주소', '좌표정보(X)', '좌표정보(Y)']

df = df[cols].copy()

print(df.shape)
df.head()

(677839, 9)


,사업장명,업태구분명,업종구분,영업상태명,인허가일자,폐업일자,도로명주소,좌표정보(X),좌표정보(Y)
0,차카라,한식,일반음식점,폐업,2001-05-23,2007-02-07,NaN,198810.765919712,448800.973307563
1,연희식당,한식,일반음식점,폐업,1996-05-13,2008-09-29,NaN,187383.887301783,444494.697447518
2,플루토,경양식,일반음식점,영업/정상,2007-08-27,,"서울특별시 종로구 사직로8길 34 (내수동,경희궁의아침3단지 지하115~119호)",197465.223921125,452366.613899898
3,무등산,한식,일반음식점,영업/정상,1995-04-14,,서울특별시 성동구 성덕정길 150 (성수동2가),205222.486515292,448231.135811849
4,청원,분식,일반음식점,폐업,2001-04-17,2002-04-19,NaN,208708.742863102,455028.589733311


In [13]:
# 날짜 변환
df['인허가일자'] = pd.to_datetime(df['인허가일자'], errors = 'coerce')
df['폐업일자'] = pd.to_datetime(df['폐업일자'], errors ='coerce')

# 도로명주소 구 이름 추출
df['구명'] = df['도로명주소'].str.extract(r'서울특별시\s+(\S+구)')
df['폐업여부'] = (df['영업상태명'] == '폐업').astype(int)
# 폐업여부 0/1 로 변환
print(f'구 추출 성공: {df["구명"].notna().sum():,}개')
print(f'구 추출 실패: {df["구명"].isna().sum():,}개')
df.head()

구 추출 성공: 386,214개
구 추출 실패: 291,625개


,사업장명,업태구분명,업종구분,영업상태명,인허가일자,폐업일자,도로명주소,좌표정보(X),좌표정보(Y),구명,폐업여부
0,차카라,한식,일반음식점,폐업,2001-05-23,2007-02-07,NaN,198810.765919712,448800.973307563,NaN,1
1,연희식당,한식,일반음식점,폐업,1996-05-13,2008-09-29,NaN,187383.887301783,444494.697447518,NaN,1
2,플루토,경양식,일반음식점,영업/정상,2007-08-27,NaT,"서울특별시 종로구 사직로8길 34 (내수동,경희궁의아침3단지 지하115~119호)",197465.223921125,452366.613899898,종로구,0
3,무등산,한식,일반음식점,영업/정상,1995-04-14,NaT,서울특별시 성동구 성덕정길 150 (성수동2가),205222.486515292,448231.135811849,성동구,0
4,청원,분식,일반음식점,폐업,2001-04-17,2002-04-19,NaN,208708.742863102,455028.589733311,NaN,1


In [20]:
# 서울 데이터 만 남기기
df_seoul = df[df['구명'].notna()].copy()

print(f'서울 데이터: {len(df_seoul):,}개')
print(f'제거된 데이터: {len(df) - len(df_seoul):,}개')
print()
print('구별 분포:')
print(df_seoul['구명'].value_counts())

NameError: name 'df_food' is not defined

In [19]:
# 인허가일자에서 연도 추출
df_recent['연도'] = df_recent['인허가일자'].dt.year

# 업태구분명 종류 확인
print(df_recent['업태구분명'].value_counts().head(15))


NameError: name 'df_recent' is not defined

In [16]:
# 업종명 매핑
업종_매핑 = {
    '한식': '한식음식점',
    '커피숍': '커피-음료',
    '경양식': '양식음식점',
    '일식': '일식음식점',
    '중국식': '중식음식점',
    '호프/통닭': '치킨전문점',
    '분식': '분식전문점'
}

# 매핑된 업종만 필터링 + 이름 통일
df_recent = df_recent[df_recent['업태구분명'].isin(업종_매핑.keys())].copy()
df_recent['업종'] = df_recent['업태구분명'].map(업종_매핑)

print(f'필터링 후: {len(df_recent):,}행')
print(df_recent['업종'].value_counts())

NameError: name 'df_recent' is not defined

In [17]:
# 인허가일자에서 연도 추출
df_recent['연도'] = pd.to_datetime(df_recent['인허가일자']).dt.year

# 구 × 연도 × 업종별 폐업률 계산
closure_detail = df_recent.groupby(['구명', '연도', '업종']).agg(
    총개수=('폐업여부', 'count'),
    폐업수=('폐업여부', 'sum')
).reset_index()

closure_detail['폐업률'] = (closure_detail['폐업수'] / closure_detail['총개수'] * 100).round(2)

# 샘플 수 너무 적은 것 제거
closure_detail = closure_detail[closure_detail['총개수'] >= 5]

print(f'총 조합 수: {len(closure_detail):,}개')
print(closure_detail.head(10))

NameError: name 'df_recent' is not defined

In [ ]:
closure_detail.to_csv('폐업률_구별_연도별_업종별.csv', index=False, encoding='utf-8-sig')
print('저장 완료!')

In [18]:
df_closed = df_recent[df_recent['폐업여부'] == 1].copy()
df_closed['생존_일수'] = (df_closed['폐업일자'] - df_closed['인허가일자']).dt.days
df_closed['생존_년수'] = (df_closed['생존_일수'] / 365).round(1)

# 업종별 평균 생존 년수
df_survival_year = df_closed.groupby('업종').agg(
    평균_생존_년수=('생존_년수', 'mean')
).reset_index()
df_survival_year['평균_생존_년수'] = df_survival_year['평균_생존_년수'].round(1)

print(df_survival_year.sort_values('평균_생존_년수', ascending=False))

NameError: name 'df_recent' is not defined

In [ ]:
import pandas as pd

# 원본 파일 직접 읽기
df1 = pd.read_csv('C:/Users/human/미정프로젝트/서울시 일반음식점 인허가 정보.csv',
                  encoding='cp949', encoding_errors='replace', low_memory=False)
df2 = pd.read_csv('C:/Users/human/미정프로젝트/서울시 휴게음식점 인허가 정보.csv',
                  encoding='cp949', encoding_errors='replace', low_memory=False)

df1['업종구분'] = '일반음식점'
df2['업종구분'] = '휴게음식점'
df_all = pd.concat([df1, df2], ignore_index=True)

# 필요한 컬럼만
cols = ['사업장명', '업태구분명', '업종구분', '영업상태명', '인허가일자', '폐업일자', '도로명주소']
df_all = df_all[cols].copy()

df_all['인허가일자'] = pd.to_datetime(df_all['인허가일자'], errors='coerce')
df_all['폐업일자'] = pd.to_datetime(df_all['폐업일자'], errors='coerce')
df_all['구명'] = df_all['도로명주소'].str.extract(r'서울특별시\s+(\S+구)')
df_all['폐업여부'] = (df_all['영업상태명'] == '폐업').astype(int)
df_all = df_all[df_all['구명'].notna()].copy()

print(f'행 수: {len(df_all):,}개')

In [ ]:
업종_매핑 = {
    '한식': '한식음식점',
    '커피숍': '커피-음료',
    '경양식': '양식음식점',
    '일식': '일식음식점',
    '중국식': '중식음식점',
    '호프/통닭': '치킨전문점',
    '분식': '분식전문점',
}

df_all = df_all[df_all['업태구분명'].isin(업종_매핑.keys())].copy()
df_all['업종'] = df_all['업태구분명'].map(업종_매핑)

df_closed_all = df_all[df_all['폐업여부'] == 1].copy()
df_closed_all['생존_년수'] = ((df_closed_all['폐업일자'] - df_closed_all['인허가일자']).dt.days / 365).round(1)

df_survival_year = df_closed_all.groupby('업종').agg(
    평균_생존_년수=('생존_년수', 'mean')
).reset_index()
df_survival_year['평균_생존_년수'] = df_survival_year['평균_생존_년수'].round(1)

print(df_survival_year.sort_values('평균_생존_년수', ascending=False))

In [ ]:
# 폐업률 파일 불러오기
df_closure = pd.read_csv('폐업률_구별_연도별_업종별.csv', encoding='utf-8-sig')

# 생존율 추가
df_closure['생존율'] = (100 - df_closure['폐업률']).round(2)

# 업종별 연도별 평균 생존율
df_survival = df_closure.groupby(['업종', '연도']).agg(
    평균_생존율=('생존율', 'mean')
).reset_index()

df_survival['평균_생존율'] = df_survival['평균_생존율'].round(2)

print(df_survival.pivot(index='업종', columns='연도', values='평균_생존율'))

In [ ]:
df_survival = df_survival[df_survival['연도'] <= 2024]

print(df_survival.pivot(index='업종', columns='연도', values='평균_생존율'))

In [ ]:
df_survival.to_csv('업종별_연도별_생존율.csv', index=False, encoding='utf-8-sig')
df_survival_year.to_csv('업종별_평균생존년수.csv', index=False, encoding='utf-8-sig')
print('저장 완료!')

In [ ]:
# 업종 매핑
업종_매핑 = {
    '한식': '한식음식점',
    '커피숍': '커피-음료',
    '경양식': '양식음식점',
    '일식': '일식음식점',
    '중국식': '중식음식점',
    '호프/통닭': '치킨전문점',
    '분식': '분식전문점',
}

df_food = df_seoul[df_seoul['업태구분명'].isin(업종_매핑.keys())].copy()
df_food['업종'] = df_food['업태구분명'].map(업종_매핑)
df_food['폐업여부'] = (df_food['영업상태명'] == '폐업').astype(int)

results = []

for year in range(2019, 2025):
    # 해당 연도에 영업 중인 가게
    # 조건: 인허가일자 <= 해당연도 AND (폐업일자 > 해당연도 OR 폐업일자 없음)
    영업중 = df_food[
        (df_food['인허가일자'].dt.year <= year) &
        ((df_food['폐업일자'].dt.year > year) | (df_food['폐업일자'].isna()))
    ].copy()
    영업중['연도'] = year
    영업중['해당연도_폐업'] = 0

    # 해당 연도에 폐업한 가게
    폐업 = df_food[df_food['폐업일자'].dt.year == year].copy()
    폐업['연도'] = year
    폐업['해당연도_폐업'] = 1

    results.append(영업중)
    results.append(폐업)

df_yearly = pd.concat(results, ignore_index=True)

# 구 × 연도 × 업종별 폐업률
closure_detail = df_yearly.groupby(['구명', '연도', '업종']).agg(
    총개수=('해당연도_폐업', 'count'),
    폐업수=('해당연도_폐업', 'sum')
).reset_index()

closure_detail['폐업률'] = (closure_detail['폐업수'] / closure_detail['총개수'] * 100).round(2)
closure_detail = closure_detail[closure_detail['총개수'] >= 5]

print(f'총 조합 수: {len(closure_detail):,}개')
print(closure_detail.head(10))